# Notebook 07 — Full-Well Model Inference

**Purpose:** Apply the trained, calibrated model from NB06 to *all* candidates in the full-well candidate union (not just QA crop windows). Produces a deployment-ready predictions parquet for every well.

**Inputs (auto-discovered from manifests + tables/):**
- `artifacts/manifests/*_model_manifest.json` — NB06 manifest
- `artifacts/manifests/*_run_manifest.json`   — NB03/04 manifest
- `tables/{nb03_run_id}_candidate_union.parquet`
- `tables/{nb03_run_id}_candidate_features.parquet`
- `artifacts/models/{nb06_run_id}_{primary_model_id}_calibrated.pkl`
- `artifacts/models/{nb06_run_id}_feature_cols.pkl`
- `artifacts/models/{nb06_run_id}_clip_bounds.pkl`
- `artifacts/models/{nb06_run_id}_train_medians.pkl`

**Outputs:**
- `tables/{RUN_ID}_fullwell_predictions.parquet`
- `artifacts/manifests/{RUN_ID}_fullwell_inference_manifest.json`
- `artifacts/reports/nb07_fullwell_inference/{RUN_ID}_overlay_{well_id}.png` (optional)

**QC cell:** Restricts full-well predictions to the NB06 QA crops, re-computes the same metric suite, and diffs against NB06 reported scores. A match confirms the globally-applied model produces identical results on cropped regions. Any delta flags a preprocessing or bgscale parity issue.

In [1]:
# ── REPO DISCOVERY ────────────────────────────────────────────────────────────
from pathlib import Path

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / '.git').exists() or (p / 'registries').exists() or (p / 'tables').exists():
            return p
    raise RuntimeError('Could not locate repo root.')

REPO_ROOT = find_repo_root()
print('REPO_ROOT =', REPO_ROOT)

REPO_ROOT = /Users/baghnatios/Desktop/spot-detection-meta


In [2]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CFG = {
    # Directory layout
    'tables_dir':   'tables',
    'models_dir':   'artifacts/models',
    'manifest_dir': 'artifacts/manifests',
    'report_dir':   'artifacts/reports/nb07_fullwell_inference',

    # Explicit run-ID bindings (None = pick latest manifest)
    'nb03_run_id': None,  # e.g. 'nb03_20260124T024537Z_066ded0c'
    'nb06_run_id': None,  # e.g. 'nb06_20260125T103012Z_aabbccdd'

    # Path overrides (None = resolve from manifests)
    'candidate_union_path_override':       None,
    'candidate_features_path_override':    None,
    'model_manifest_path_override':        None,
    'nb06_test_predictions_path_override': None,

    # Model selection (None = use primary_model_id from NB06 manifest)
    'model_id_override': None,

    # Ensemble: score all saved models + add mean column
    'run_ensemble': False,

    # Overlays
    'write_overlays':           True,
    'overlay_max_wells':        4,
    'overlay_lo_pct':           1,
    'overlay_hi_pct':           99,
    'overlay_circle_radius_px': 10,
    'data_root_override':       None,  # e.g. '/Users/me/Desktop/spot_detection'

    # QC crop comparison splits
    'qc_splits_to_check': ['test', 'calibration', 'train'],

    # Provenance
    'inference_registry_version': 'v1_nb07_fullwell',
    'random_seed': 42,
}
print('Config OK.')

Config OK.


In [3]:
# ── IMPORTS ───────────────────────────────────────────────────────────────────
import gc, hashlib, json, math, os, re, time, warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import tifffile

from sklearn.metrics import (
    average_precision_score, brier_score_loss,
    f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix,
    precision_recall_curve, roc_curve,
)

print('Imports OK.')

Imports OK.


In [4]:
# ── HELPERS ───────────────────────────────────────────────────────────────────
def ts_utc():
    return datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')

def log(msg):
    print(f'[{ts_utc()}] {msg}', flush=True)

def ensure_dir(path):
    p = Path(path); p.mkdir(parents=True, exist_ok=True); return p

def make_run_id(prefix='nb07'):
    t = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    h = hashlib.sha1(f'{t}|{os.getpid()}|nb07'.encode()).hexdigest()[:8]
    return f'{prefix}_{t}_{h}'

def safe_to_parquet(df, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(path, index=False)
    except Exception as e:
        csv = path.with_suffix('.csv'); df.to_csv(csv, index=False)
        log(f'[warn] parquet failed ({e}); wrote CSV: {csv.name}'); return csv
    return path

def write_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str), encoding='utf-8')
    return path

def latest_file(directory, patterns):
    candidates = []
    if not directory.exists(): return None
    for pat in patterns:
        candidates.extend(directory.glob(pat))
    candidates = [p for p in candidates if p.is_file()]
    if not candidates: return None
    candidates.sort(key=lambda p: (p.stat().st_mtime, str(p)), reverse=True)
    return candidates[0]

def eval_metrics(y_true, y_prob, threshold=0.5, label=''):
    y_pred = (y_prob >= threshold).astype(int)
    out = {'label': label, 'n': int(len(y_true)), 'n_pos': int(y_true.sum()),
           'threshold': float(threshold)}
    try:    out['roc_auc']       = float(roc_auc_score(y_true, y_prob))
    except: out['roc_auc']       = float('nan')
    try:    out['avg_precision'] = float(average_precision_score(y_true, y_prob))
    except: out['avg_precision'] = float('nan')
    try:    out['brier']         = float(brier_score_loss(y_true, y_prob))
    except: out['brier']         = float('nan')
    out['f1']        = float(f1_score(y_true, y_pred, zero_division=0))
    out['precision'] = float(precision_score(y_true, y_pred, zero_division=0))
    out['recall']    = float(recall_score(y_true, y_pred, zero_division=0))
    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        out['tn'] = int(tn); out['fp'] = int(fp)
        out['fn'] = int(fn); out['tp'] = int(tp)
    except:
        out['tn'] = out['fp'] = out['fn'] = out['tp'] = 0
    return out

RUN_ID       = make_run_id('nb07')
TABLES_DIR   = ensure_dir(REPO_ROOT / CFG['tables_dir'])
MODELS_DIR   = ensure_dir(REPO_ROOT / CFG['models_dir'])
MANIFEST_DIR = ensure_dir(REPO_ROOT / CFG['manifest_dir'])
REPORT_DIR   = ensure_dir(REPO_ROOT / CFG['report_dir'])
log(f'RUN_ID = {RUN_ID}')

[2026-01-25 03:27:05 UTC] RUN_ID = nb07_20260125T032705Z_c3eff871


## 1. Load NB06 model manifest and artifacts

In [5]:
# ── DISCOVER NB06 MANIFEST ────────────────────────────────────────────────────
log('=' * 70)
log('STEP 1: Discovering NB06 model manifest ...')

nb06_manifest_path = None
if CFG.get('model_manifest_path_override'):
    nb06_manifest_path = Path(CFG['model_manifest_path_override'])
    assert nb06_manifest_path.exists()
elif CFG.get('nb06_run_id'):
    for suffix in ['_model_manifest.json', '_manifest.json']:
        _p = MANIFEST_DIR / f"{CFG['nb06_run_id']}{suffix}"
        if _p.exists(): nb06_manifest_path = _p; break
    assert nb06_manifest_path, f"No manifest for nb06_run_id={CFG['nb06_run_id']}"
else:
    nb06_manifest_path = latest_file(MANIFEST_DIR, ['*model_manifest.json'])
    assert nb06_manifest_path, 'No *model_manifest.json found. Run NB06 first.'

nb06_manifest    = json.loads(nb06_manifest_path.read_text(encoding='utf-8'))
NB06_RUN_ID      = nb06_manifest['run_id']
PRIMARY_MODEL_ID = CFG.get('model_id_override') or nb06_manifest['primary_model_id']
DECISION_THRESHOLD = float(nb06_manifest['thresholds'][PRIMARY_MODEL_ID])
THRESHOLD_POLICY   = nb06_manifest.get('threshold_policy', 'max_f1')
NB06_REPORTED      = nb06_manifest.get('test_metrics', {}).get(PRIMARY_MODEL_ID, {})

log(f'  NB06 manifest  : {nb06_manifest_path.name}')
log(f'  Primary model  : {PRIMARY_MODEL_ID}')
log(f'  Threshold      : {DECISION_THRESHOLD:.6f}  (policy={THRESHOLD_POLICY})')
log(f'  NB06 test AP   : {NB06_REPORTED.get("avg_precision", "n/a")}')
log(f'  NB06 test F1   : {NB06_REPORTED.get("f1", "n/a")}')
log(f'  NB06 test AUC  : {NB06_REPORTED.get("roc_auc", "n/a")}')

[2026-01-25 03:27:05 UTC] ======================================================================
[2026-01-25 03:27:05 UTC] STEP 1: Discovering NB06 model manifest ...
[2026-01-25 03:27:05 UTC]   NB06 manifest  : nb06_20260124T172633Z_5cf12d26_model_manifest.json
[2026-01-25 03:27:05 UTC]   Primary model  : lgbm_tuned
[2026-01-25 03:27:05 UTC]   Threshold      : 0.600000  (policy=max_f1)
[2026-01-25 03:27:05 UTC]   NB06 test AP   : 0.8142453275102429
[2026-01-25 03:27:05 UTC]   NB06 test F1   : 0.8314606741573034
[2026-01-25 03:27:05 UTC]   NB06 test AUC  : 0.8771306818181819


In [6]:
# ── LOAD MODEL ARTIFACTS ──────────────────────────────────────────────────────
log('Loading model artifacts ...')

def _artifact(run_id, suffix):
    exact = MODELS_DIR / f'{run_id}_{suffix}'
    if exact.exists(): return exact
    hits = sorted(MODELS_DIR.glob(f'*_{suffix}'),
                  key=lambda p: p.stat().st_mtime, reverse=True)
    return hits[0] if hits else None

cal_model_path     = _artifact(NB06_RUN_ID, f'{PRIMARY_MODEL_ID}_calibrated.pkl')
feature_cols_path  = _artifact(NB06_RUN_ID, 'feature_cols.pkl')
clip_bounds_path   = _artifact(NB06_RUN_ID, 'clip_bounds.pkl')
train_medians_path = _artifact(NB06_RUN_ID, 'train_medians.pkl')

for name, p in [('model',          cal_model_path),
                ('feature_cols',   feature_cols_path),
                ('clip_bounds',    clip_bounds_path),
                ('train_medians',  train_medians_path)]:
    assert p and p.exists(), f'Missing artifact: {name}'
    log(f'  {name:15s}: {p.name}')

CAL_MODEL     = joblib.load(cal_model_path)
FEATURE_COLS  = joblib.load(feature_cols_path)
CLIP_BOUNDS   = joblib.load(clip_bounds_path)
TRAIN_MEDIANS = joblib.load(train_medians_path)

log(f'  Feature cols   : {len(FEATURE_COLS)}')

OTHER_MODELS = {}
if CFG['run_ensemble']:
    for mid in nb06_manifest.get('thresholds', {}):
        if mid == PRIMARY_MODEL_ID: continue
        p = _artifact(NB06_RUN_ID, f'{mid}_calibrated.pkl')
        if p and p.exists():
            OTHER_MODELS[mid] = {'model': joblib.load(p),
                                 'threshold': float(nb06_manifest['thresholds'][mid])}
            log(f'  ensemble [{mid}]')

log('Model artifacts loaded.')

[2026-01-25 03:27:05 UTC] Loading model artifacts ...
[2026-01-25 03:27:05 UTC]   model          : nb06_20260124T172633Z_5cf12d26_lgbm_tuned_calibrated.pkl
[2026-01-25 03:27:05 UTC]   feature_cols   : nb06_20260124T172633Z_5cf12d26_feature_cols.pkl
[2026-01-25 03:27:05 UTC]   clip_bounds    : nb06_20260124T172633Z_5cf12d26_clip_bounds.pkl
[2026-01-25 03:27:05 UTC]   train_medians  : nb06_20260124T172633Z_5cf12d26_train_medians.pkl
[2026-01-25 03:27:05 UTC]   Feature cols   : 249
[2026-01-25 03:27:05 UTC] Model artifacts loaded.


## 2. Load NB03/04 candidate union + features

In [7]:
# ── DISCOVER NB03 ARTIFACTS ───────────────────────────────────────────────────
log('=' * 70)
log('STEP 2: Discovering NB03/04 artifacts ...')

NB03_RUN_ID   = CFG.get('nb03_run_id')
nb03_manifest = None

# Try to infer NB03 run_id from NB06 manifest inputs
if NB03_RUN_ID is None:
    for v in nb06_manifest.get('inputs', {}).values():
        m = re.match(r'.*(nb03_\w+)_candidate', str(v))
        if m: NB03_RUN_ID = m.group(1); break

if NB03_RUN_ID:
    for suffix in ['_run_manifest.json', '_manifest.json']:
        _p = MANIFEST_DIR / f'{NB03_RUN_ID}{suffix}'
        if _p.exists():
            nb03_manifest = json.loads(_p.read_text(encoding='utf-8'))
            log(f'  NB03 manifest  : {_p.name}')
            break

def _resolve(override_key, manifest_key, glob_patterns):
    if CFG.get(override_key):
        p = Path(CFG[override_key]); assert p.exists(); return p
    if nb03_manifest:
        mp = nb03_manifest.get('paths', {}).get(manifest_key)
        if mp and Path(mp).exists(): return Path(mp)
    if NB03_RUN_ID:
        for g in glob_patterns:
            p = latest_file(TABLES_DIR, [f'{NB03_RUN_ID}_{g}'])
            if p: return p
    return latest_file(TABLES_DIR, glob_patterns)

union_path    = _resolve('candidate_union_path_override',
                         'candidate_union',    ['*candidate_union.parquet'])
features_path = _resolve('candidate_features_path_override',
                         'candidate_features', ['*candidate_features.parquet'])

assert union_path    and union_path.exists(),    'candidate_union not found'
assert features_path and features_path.exists(), 'candidate_features not found'

log(f'  candidate_union   : {union_path.name}')
log(f'  candidate_features: {features_path.name}')

t0 = time.perf_counter()
union_df    = pd.read_parquet(union_path)
features_df = pd.read_parquet(features_path)
log(f'  Loaded in {time.perf_counter()-t0:.1f}s')
log(f'  union_df    : {len(union_df):,} rows')
log(f'  features_df : {len(features_df):,} rows x {len(features_df.columns)} cols')
log(f'  Wells       : {sorted(union_df["well_id"].astype(str).unique().tolist())}')

[2026-01-25 03:27:05 UTC] ======================================================================
[2026-01-25 03:27:05 UTC] STEP 2: Discovering NB03/04 artifacts ...
[2026-01-25 03:27:05 UTC]   candidate_union   : nb03_20260124T024537Z_066ded0c_candidate_union.parquet
[2026-01-25 03:27:05 UTC]   candidate_features: nb03_20260124T024537Z_066ded0c_candidate_features.parquet
[2026-01-25 03:27:06 UTC]   Loaded in 0.2s
[2026-01-25 03:27:06 UTC]   union_df    : 5,955 rows
[2026-01-25 03:27:06 UTC]   features_df : 5,955 rows x 434 cols
[2026-01-25 03:27:06 UTC]   Wells       : ['C8', 'D8']


## 3. Build feature matrix and run inference

In [8]:
# ── MERGE + PREPROCESS ────────────────────────────────────────────────────────
log('=' * 70)
log('STEP 3: Building feature matrix ...')

merged = union_df.merge(features_df, on='union_id', how='left', suffixes=('', '_feat'))
merged = merged.drop(columns=[c for c in merged.columns if c.endswith('_feat')])
log(f'  Merged: {len(merged):,} rows')

missing_cols = [c for c in FEATURE_COLS if c not in merged.columns]
if missing_cols:
    log(f'  [warn] {len(missing_cols)} feature cols missing — will be imputed as NaN')
    for c in missing_cols:
        merged[c] = float('nan')
else:
    log(f'  All {len(FEATURE_COLS)} feature cols present.')

# Apply identical clip + impute as NB06
X_all = merged[FEATURE_COLS].copy()

for col in FEATURE_COLS:
    lo, hi = CLIP_BOUNDS.get(col, (float('nan'), float('nan')))
    if lo is not None and hi is not None and np.isfinite(lo) and np.isfinite(hi):
        X_all[col] = X_all[col].clip(lower=lo, upper=hi)

for col in FEATURE_COLS:
    if hasattr(TRAIN_MEDIANS, 'get'):
        med = TRAIN_MEDIANS.get(col, float('nan'))
    else:
        med = float(TRAIN_MEDIANS[col]) if col in TRAIN_MEDIANS.index else float('nan')
    med = float(med) if med is not None else float('nan')
    X_all[col] = X_all[col].fillna(med if np.isfinite(med) else 0.0)

X_np = X_all.values.astype(np.float32)
# Safety: replace any remaining NaN with 0
X_np = np.where(np.isnan(X_np), 0.0, X_np)
log(f'  X_np shape: {X_np.shape}')
log(f'  NaN remaining: {int(np.isnan(X_np).sum())} (should be 0)')

[2026-01-25 03:27:06 UTC] ======================================================================
[2026-01-25 03:27:06 UTC] STEP 3: Building feature matrix ...
[2026-01-25 03:27:06 UTC]   Merged: 5,955 rows
[2026-01-25 03:27:06 UTC]   All 249 feature cols present.
[2026-01-25 03:27:06 UTC]   X_np shape: (5955, 249)
[2026-01-25 03:27:06 UTC]   NaN remaining: 0 (should be 0)


In [9]:
# ── RUN PRIMARY INFERENCE ─────────────────────────────────────────────────────
log('Running primary model inference on full-well candidates ...')
t0 = time.perf_counter()

prob_cal  = CAL_MODEL.predict_proba(X_np)[:, 1].astype(np.float32)
predicted = (prob_cal >= DECISION_THRESHOLD).astype(np.int8)

log(f'  Inference time: {time.perf_counter()-t0:.1f}s')
log(f'  Candidates    : {len(prob_cal):,}')
log(f'  Predicted pos : {int(predicted.sum()):,}  ({100*predicted.mean():.2f}%)')
log(f'  Prob range    : [{prob_cal.min():.4f}, {prob_cal.max():.4f}]  '
    f'median={float(np.median(prob_cal)):.4f}')

ensemble_prob = None
if CFG['run_ensemble'] and OTHER_MODELS:
    all_probs = [prob_cal]
    for mid, minfo in OTHER_MODELS.items():
        p = minfo['model'].predict_proba(X_np)[:, 1].astype(np.float32)
        all_probs.append(p)
        log(f'  ensemble [{mid}]: pos={int((p >= minfo["threshold"]).sum()):,}')
    ensemble_prob = np.stack(all_probs, axis=0).mean(axis=0).astype(np.float32)

[2026-01-25 03:27:06 UTC] Running primary model inference on full-well candidates ...
[2026-01-25 03:27:06 UTC]   Inference time: 0.1s
[2026-01-25 03:27:06 UTC]   Candidates    : 5,955
[2026-01-25 03:27:06 UTC]   Predicted pos : 2,872  (48.23%)
[2026-01-25 03:27:06 UTC]   Prob range    : [0.0000, 1.0000]  median=0.3333


/Users/baghnatios/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [10]:
# ── ASSEMBLE PREDICTIONS TABLE ────────────────────────────────────────────────
KEEP_COLS = [
    'union_id', 'run_id', 'dataset_id', 'well_id', 'crop_id',
    'union_centroid_well_y_px', 'union_centroid_well_x_px',
    'union_medoid_well_y_px',   'union_medoid_well_x_px',
    'union_n_members', 'union_radius_px',
    'top_method_id', 'top_method_score_raw', 'top_method_score_norm',
    'proposer_support_count', 'proposer_support_fraction',
    'family_support_count',   'family_set_canonical',
]
keep = [c for c in KEEP_COLS if c in merged.columns]
preds_df = merged[keep].copy()

preds_df['prob_true_spot']             = prob_cal
preds_df['predicted_label']            = predicted
preds_df['decision_threshold']         = float(DECISION_THRESHOLD)
preds_df['threshold_policy']           = THRESHOLD_POLICY
preds_df['model_id']                   = PRIMARY_MODEL_ID
preds_df['model_nb06_run_id']          = NB06_RUN_ID
preds_df['inference_registry_version'] = CFG['inference_registry_version']
preds_df['nb07_run_id']                = RUN_ID
preds_df['created_at_utc']             = datetime.now(timezone.utc).isoformat()

if ensemble_prob is not None:
    preds_df['prob_true_spot_ensemble']  = ensemble_prob
    preds_df['predicted_label_ensemble'] = (ensemble_prob >= DECISION_THRESHOLD).astype(np.int8)

log(f'preds_df: {len(preds_df):,} rows x {len(preds_df.columns)} cols')

well_summary = (
    preds_df.groupby('well_id')
    .agg(n_candidates=('union_id', 'count'),
         n_predicted=('predicted_label', 'sum'),
         prob_mean=('prob_true_spot', 'mean'),
         prob_p90=('prob_true_spot', lambda x: float(np.percentile(x, 90))))
    .reset_index()
)
well_summary['pred_rate_pct'] = (100 * well_summary['n_predicted'] / well_summary['n_candidates']).round(2)
display(well_summary)

[2026-01-25 03:27:06 UTC] preds_df: 5,955 rows x 27 cols


,well_id,n_candidates,n_predicted,prob_mean,prob_p90,pred_rate_pct
0,C8,2390,710,0.287261,0.952381,29.71
1,D8,3565,2162,0.579397,0.952381,60.65


## 4. QC — Re-score GT crops and compare vs NB06

This is the parity check. We pull the same QA annotations used in NB06, look up those union_ids in the **full-well** predictions table, and recompute the exact same metric suite.

- If scores match within tolerance → the full-well feature distributions are identical to the crop-scoped NB06 run. ✅
- If scores differ → something changed (bgscale normalization, union merging, imputation). ⚠️

In [11]:
# ── LOAD QC INPUTS ────────────────────────────────────────────────────────────
log('=' * 70)
log('STEP 4 QC: Loading NB05/06 supervision artifacts ...')

sup_path    = latest_file(TABLES_DIR, ['*training_supervision_table.parquet'])
splits_path = latest_file(TABLES_DIR, ['*splits.parquet'])

# NB06 test predictions parquet (for the reference curve overlay)
_report_dir_nb06 = REPO_ROOT / 'artifacts' / 'reports' / 'nb06_model'
_nb06_test_pred_path = None
if CFG.get('nb06_test_predictions_path_override'):
    _nb06_test_pred_path = Path(CFG['nb06_test_predictions_path_override'])
else:
    _nb06_test_pred_path = latest_file(_report_dir_nb06, ['*test_predictions_errors.parquet'])
    if _nb06_test_pred_path is None:
        # Try manifest outputs
        _tp = nb06_manifest.get('outputs', {}).get('test_predictions_errors')
        if _tp and Path(_tp).exists():
            _nb06_test_pred_path = Path(_tp)

log(f'  supervision table : {sup_path.name if sup_path else "NOT FOUND"}')
log(f'  splits table      : {splits_path.name if splits_path else "NOT FOUND"}')
log(f'  nb06 test preds   : {_nb06_test_pred_path.name if _nb06_test_pred_path else "NOT FOUND (delta comparison skipped)"}')

qc_sup    = pd.read_parquet(sup_path)    if sup_path    and sup_path.exists()    else pd.DataFrame()
qc_splits = pd.read_parquet(splits_path) if splits_path and splits_path.exists() else pd.DataFrame()
qc_nb06   = (pd.read_parquet(_nb06_test_pred_path)
             if _nb06_test_pred_path and _nb06_test_pred_path.exists()
             else pd.DataFrame())

for name, df in [('supervision', qc_sup), ('splits', qc_splits), ('nb06_test_preds', qc_nb06)]:
    log(f'  loaded {name}: {len(df):,} rows')

qc_eval = pd.DataFrame()

[2026-01-25 03:27:06 UTC] ======================================================================
[2026-01-25 03:27:06 UTC] STEP 4 QC: Loading NB05/06 supervision artifacts ...
[2026-01-25 03:27:06 UTC]   supervision table : nb05_20260124T042657Z_2af2592e_training_supervision_table.parquet
[2026-01-25 03:27:06 UTC]   splits table      : nb05_20260124T042657Z_2af2592e_splits.parquet
[2026-01-25 03:27:06 UTC]   nb06 test preds   : nb06_20260124T172633Z_5cf12d26_test_predictions_errors.parquet
[2026-01-25 03:27:06 UTC]   loaded supervision: 864 rows
[2026-01-25 03:27:06 UTC]   loaded splits: 50 rows
[2026-01-25 03:27:06 UTC]   loaded nb06_test_preds: 92 rows


In [12]:
# ── QC CORE: RE-SCORE GT CROPS USING FULL-WELL PREDICTIONS ───────────────────
log('QC: Re-scoring GT annotations using full-well predictions ...')

QC_RESULTS = {}

if len(qc_sup) == 0 or len(qc_splits) == 0:
    log('[skip] Missing supervision or splits — QC skipped.')
else:
    qc_sup['union_id']    = qc_sup['union_id'].astype(str)
    qc_splits['crop_id']  = qc_splits['crop_id'].astype(str)
    preds_df['union_id']  = preds_df['union_id'].astype(str)

    # Same supervision filter as NB06
    if 'supervision_status' in qc_sup.columns:
        qc_labeled = qc_sup[qc_sup['supervision_status'] == 'included'].copy()
    else:
        qc_labeled = qc_sup.copy()

    qc_labeled['target_binary'] = pd.to_numeric(
        qc_labeled['target_binary'], errors='coerce'
    ).fillna(0.0)

    # Join split role
    qc_labeled = qc_labeled.drop(
        columns=[c for c in ['split_role', 'split_id'] if c in qc_labeled.columns]
    ).merge(qc_splits[['crop_id', 'split_role']], on='crop_id', how='left')

    # THE KEY STEP: join FULL-WELL probabilities (from NB07) onto supervision rows
    fw_probs = preds_df[['union_id', 'prob_true_spot', 'predicted_label']].rename(
        columns={'prob_true_spot': 'fw_prob', 'predicted_label': 'fw_predicted'}
    )
    qc_joined = qc_labeled.merge(fw_probs, on='union_id', how='left')

    n_matched = int(qc_joined['fw_prob'].notna().sum())
    n_missing = int(qc_joined['fw_prob'].isna().sum())
    log(f'  Supervision rows matched to fw preds: {n_matched:,}')
    if n_missing > 0:
        log(f'  [warn] {n_missing} supervision rows have no fw prediction '
            '(candidate dropped by NMS/union or different NB03 run?)')

    qc_eval = qc_joined[qc_joined['fw_prob'].notna()].copy()
    log(f'  Evaluable rows: {len(qc_eval):,}  '
        f'(pos={int(qc_eval["target_binary"].sum())}, '
        f'neg={int((qc_eval["target_binary"]==0).sum())})')

    for split_role in CFG['qc_splits_to_check']:
        df_s = qc_eval[qc_eval['split_role'] == split_role].copy()
        if len(df_s) == 0:
            log(f'  [{split_role}] 0 rows — skip')
            continue
        y_true = df_s['target_binary'].values
        y_prob = df_s['fw_prob'].values
        if len(np.unique(y_true)) < 2:
            log(f'  [{split_role}] only one class — skip')
            continue
        m = eval_metrics(y_true, y_prob, threshold=DECISION_THRESHOLD, label=f'fw_{split_role}')
        QC_RESULTS[split_role] = m
        log(f'  [{split_role:12s}] n={m["n"]} pos={m["n_pos"]}  '
            f'AP={m["avg_precision"]:.4f}  F1={m["f1"]:.4f}  '
            f'AUC={m["roc_auc"]:.4f}  Brier={m["brier"]:.4f}')

[2026-01-25 03:27:06 UTC] QC: Re-scoring GT annotations using full-well predictions ...
[2026-01-25 03:27:06 UTC]   Supervision rows matched to fw preds: 861
[2026-01-25 03:27:06 UTC]   Evaluable rows: 861  (pos=299, neg=562)
[2026-01-25 03:27:06 UTC]   [test        ] n=66 pos=31  AP=1.0000  F1=1.0000  AUC=1.0000  Brier=0.0016
[2026-01-25 03:27:06 UTC]   [calibration ] n=171 pos=16  AP=0.8233  F1=0.8485  AUC=0.9871  Brier=0.0242
[2026-01-25 03:27:06 UTC]   [train       ] n=624 pos=252  AP=0.9669  F1=0.9624  AUC=0.9841  Brier=0.0264


In [13]:
# ── QC DELTA TABLE: NB07 full-well vs NB06 reported metrics ──────────────────
log('=' * 70)
log('QC DELTA: NB07 full-well scores vs NB06 reported scores ...')

METRIC_KEYS = ['avg_precision', 'roc_auc', 'f1', 'precision', 'recall', 'brier']

if not QC_RESULTS:
    log('[skip] No QC results computed.')
else:
    delta_rows = []
    for split_role, fw_metrics in QC_RESULTS.items():
        # For the test split, reference is NB06 reported test metrics
        ref = NB06_REPORTED if split_role == 'test' else {}
        row = {'split_role': split_role}
        for k in METRIC_KEYS:
            fw  = fw_metrics.get(k, float('nan'))
            nb6 = ref.get(k, float('nan'))
            row[f'fw_{k}']    = round(fw,  4) if np.isfinite(fw)  else float('nan')
            row[f'nb06_{k}']  = round(nb6, 4) if np.isfinite(nb6) else float('nan')
            row[f'delta_{k}'] = round(fw - nb6, 5) if (np.isfinite(fw) and np.isfinite(nb6)) else float('nan')
        delta_rows.append(row)

    delta_df = pd.DataFrame(delta_rows).set_index('split_role')
    print('\n--- Delta table (fw = this NB07 run, nb06 = NB06 reported, delta = fw - nb06) ---')
    display(delta_df)

    # Parity verdict
    PARITY_TOL = 1e-4
    if 'test' in QC_RESULTS and NB06_REPORTED:
        fw_test  = QC_RESULTS['test']
        bad = []
        for k in METRIC_KEYS:
            fw  = fw_test.get(k, float('nan'))
            nb6 = NB06_REPORTED.get(k, float('nan'))
            if np.isfinite(fw) and np.isfinite(nb6) and abs(fw - nb6) > PARITY_TOL:
                bad.append((k, fw, nb6, fw - nb6))
        if not bad:
            print(f'\n✅  PARITY CONFIRMED — Full-well NB07 test scores match NB06 '
                  f'within tolerance ({PARITY_TOL}).')
        else:
            print(f'\n⚠️   PARITY MISMATCH — {len(bad)} metric(s) differ by > {PARITY_TOL}:')
            for k, fw, nb6, delta in bad:
                print(f'   {k:20s}  fw={fw:.5f}  nb06={nb6:.5f}  delta={delta:+.5f}')
            print('\n   Likely causes:')
            print('   • bgscale_* or *_global_norm features differ (full-well vs crop-scoped support)')
            print('   • Different candidate universe (union merging or NMS radius changed)')
            print('   • Feature imputation mismatch (train_medians applied to different NaN pattern)')
            print('   • Supervision table expanded (implicit negatives shifted class balance)')
    else:
        log('[info] NB06 reference metrics unavailable — parity verdict skipped.')

[2026-01-25 03:27:06 UTC] ======================================================================
[2026-01-25 03:27:06 UTC] QC DELTA: NB07 full-well scores vs NB06 reported scores ...

--- Delta table (fw = this NB07 run, nb06 = NB06 reported, delta = fw - nb06) ---


,fw_avg_precision,nb06_avg_precision,delta_avg_precision,fw_roc_auc,nb06_roc_auc,delta_roc_auc,fw_f1,nb06_f1,delta_f1,fw_precision,nb06_precision,delta_precision,fw_recall,nb06_recall,delta_recall,fw_brier,nb06_brier,delta_brier
split_role,,,,,,,,,,,,,,,,,,
test,1.0000,0.8142,0.18575,1.0000,0.8771,0.12287,1.0000,0.8315,0.16854,1.0000,0.8222,0.17778,1.0000,0.8409,0.15909,0.0016,0.1407,-0.13913
calibration,0.8233,NaN,NaN,0.9871,NaN,NaN,0.8485,NaN,NaN,0.8235,NaN,NaN,0.8750,NaN,NaN,0.0242,NaN,NaN
train,0.9669,NaN,NaN,0.9841,NaN,NaN,0.9624,NaN,NaN,0.9605,NaN,NaN,0.9643,NaN,NaN,0.0264,NaN,NaN



⚠️   PARITY MISMATCH — 6 metric(s) differ by > 0.0001:
   avg_precision         fw=1.00000  nb06=0.81425  delta=+0.18575
   roc_auc               fw=1.00000  nb06=0.87713  delta=+0.12287
   f1                    fw=1.00000  nb06=0.83146  delta=+0.16854
   precision             fw=1.00000  nb06=0.82222  delta=+0.17778
   recall                fw=1.00000  nb06=0.84091  delta=+0.15909
   brier                 fw=0.00161  nb06=0.14074  delta=-0.13913

   Likely causes:
   • bgscale_* or *_global_norm features differ (full-well vs crop-scoped support)
   • Different candidate universe (union merging or NMS radius changed)
   • Feature imputation mismatch (train_medians applied to different NaN pattern)
   • Supervision table expanded (implicit negatives shifted class balance)


In [14]:
# ── QC DIAGNOSTIC PLOTS: PR + ROC (NB07 fw vs NB06 reference curve) ──────────
if QC_RESULTS and len(qc_eval) > 0:
    log('Generating QC diagnostic plots ...')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = {'test': 'tab:blue', 'calibration': 'tab:orange', 'train': 'tab:green'}

    # PR curve
    ax = axes[0]
    for sr in CFG['qc_splits_to_check']:
        df_s = qc_eval[qc_eval['split_role'] == sr] if 'split_role' in qc_eval.columns else pd.DataFrame()
        if len(df_s) == 0 or len(np.unique(df_s['target_binary'])) < 2: continue
        p, r, _ = precision_recall_curve(df_s['target_binary'].values, df_s['fw_prob'].values)
        ap = QC_RESULTS.get(sr, {}).get('avg_precision', float('nan'))
        ax.plot(r, p, lw=2, color=colors.get(sr, 'gray'),
                label=f'NB07 fw {sr} (AP={ap:.3f})')
    # NB06 reference
    if len(qc_nb06) > 0 and 'target_binary' in qc_nb06.columns and 'prob_cal' in qc_nb06.columns:
        if len(np.unique(qc_nb06['target_binary'])) > 1:
            p_r, r_r, _ = precision_recall_curve(qc_nb06['target_binary'].values, qc_nb06['prob_cal'].values)
            ap_r = NB06_REPORTED.get('avg_precision', float('nan'))
            ax.plot(r_r, p_r, 'k--', lw=1.5, alpha=0.7, label=f'NB06 test ref (AP={ap_r:.3f})')
    # Threshold point
    if 'test' in QC_RESULTS:
        m = QC_RESULTS['test']
        ax.plot(m.get('recall', float('nan')), m.get('precision', float('nan')),
                '*', ms=14, color='tab:blue',
                label=f'threshold={DECISION_THRESHOLD:.3f}')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title('PR — NB07 full-well vs NB06 reference')
    ax.legend(fontsize=8); ax.set_xlim(0, 1); ax.set_ylim(0, 1)

    # ROC curve
    ax = axes[1]
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='random')
    for sr in CFG['qc_splits_to_check']:
        df_s = qc_eval[qc_eval['split_role'] == sr] if 'split_role' in qc_eval.columns else pd.DataFrame()
        if len(df_s) == 0 or len(np.unique(df_s['target_binary'])) < 2: continue
        fpr, tpr, _ = roc_curve(df_s['target_binary'].values, df_s['fw_prob'].values)
        auc = QC_RESULTS.get(sr, {}).get('roc_auc', float('nan'))
        ax.plot(fpr, tpr, lw=2, color=colors.get(sr, 'gray'),
                label=f'NB07 fw {sr} (AUC={auc:.3f})')
    if len(qc_nb06) > 0 and 'target_binary' in qc_nb06.columns and 'prob_cal' in qc_nb06.columns:
        if len(np.unique(qc_nb06['target_binary'])) > 1:
            fpr_r, tpr_r, _ = roc_curve(qc_nb06['target_binary'].values, qc_nb06['prob_cal'].values)
            auc_r = NB06_REPORTED.get('roc_auc', float('nan'))
            ax.plot(fpr_r, tpr_r, 'k--', lw=1.5, alpha=0.7, label=f'NB06 test ref (AUC={auc_r:.3f})')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('ROC — NB07 full-well vs NB06 reference')
    ax.legend(fontsize=8); ax.set_xlim(0, 1); ax.set_ylim(0, 1)

    fig.suptitle(
        f'QC Parity Check | model={PRIMARY_MODEL_ID} | '
        f'threshold={DECISION_THRESHOLD:.4f} | NB06 run={NB06_RUN_ID}',
        fontsize=10, y=1.02
    )
    fig.tight_layout()
    qc_plot = REPORT_DIR / f'{RUN_ID}_qc_parity_pr_roc.png'
    fig.savefig(qc_plot, dpi=160, bbox_inches='tight')
    plt.show(); plt.close(fig)
    log(f'  QC plot saved: {qc_plot.name}')

[2026-01-25 03:27:06 UTC] Generating QC diagnostic plots ...
[2026-01-25 03:27:06 UTC]   QC plot saved: nb07_20260125T032705Z_c3eff871_qc_parity_pr_roc.png


/var/folders/mf/blf3nbg52djf7wbt4080gd040000gn/T/ipykernel_10032/373909008.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close(fig)


In [15]:
# ── QC PER-CROP BREAKDOWN ─────────────────────────────────────────────────────
# For each test crop: compare per-crop AP and F1 from NB07 vs NB06.
# Large per-crop deltas pinpoint which crops changed and why.
if len(qc_eval) > 0 and 'crop_id' in qc_eval.columns and 'split_role' in qc_eval.columns:
    log('Per-crop breakdown (test split) ...')
    qc_test = qc_eval[qc_eval['split_role'] == 'test'].copy()

    if len(qc_test) > 0:
        crop_rows = []
        for crop_id, g in qc_test.groupby('crop_id'):
            y_t = g['target_binary'].values
            y_p = g['fw_prob'].values
            row = {'crop_id': crop_id,
                   'well_id': g['well_id'].iloc[0] if 'well_id' in g.columns else '',
                   'n': len(g), 'n_pos': int(y_t.sum())}
            if len(np.unique(y_t)) == 2:
                row['fw_ap'] = round(float(average_precision_score(y_t, y_p)), 4)
                row['fw_f1'] = round(float(f1_score(y_t, (y_p >= DECISION_THRESHOLD).astype(int), zero_division=0)), 4)
            else:
                row['fw_ap'] = row['fw_f1'] = float('nan')

            # NB06 reference for this crop
            if len(qc_nb06) > 0 and 'crop_id' in qc_nb06.columns:
                g6 = qc_nb06[qc_nb06['crop_id'] == crop_id]
                if len(g6) > 0 and 'target_binary' in g6.columns and 'prob_cal' in g6.columns:
                    y_t6 = g6['target_binary'].values
                    y_p6 = g6['prob_cal'].values
                    if len(np.unique(y_t6)) == 2:
                        row['nb06_ap'] = round(float(average_precision_score(y_t6, y_p6)), 4)
                        row['nb06_f1'] = round(float(f1_score(y_t6, (y_p6 >= DECISION_THRESHOLD).astype(int), zero_division=0)), 4)
                        row['delta_ap'] = round(row['fw_ap'] - row['nb06_ap'], 4) if np.isfinite(row.get('fw_ap', float('nan'))) else float('nan')
                        row['delta_f1'] = round(row['fw_f1'] - row['nb06_f1'], 4) if np.isfinite(row.get('fw_f1', float('nan'))) else float('nan')
            crop_rows.append(row)

        crop_df = pd.DataFrame(crop_rows).sort_values('fw_ap', ascending=True)
        display(crop_df)

        if 'delta_ap' in crop_df.columns:
            worst = crop_df[crop_df['delta_ap'].abs() > 0.05]
            if len(worst):
                print(f'\n⚠️  {len(worst)} crop(s) with |delta_AP| > 0.05:')
                display(worst)
            else:
                print('\n✅  All crops have |delta_AP| <= 0.05.')
    else:
        log('[info] No test-split rows in qc_eval.')

[2026-01-25 03:27:06 UTC] Per-crop breakdown (test split) ...


,crop_id,well_id,n,n_pos,fw_ap,fw_f1
0,crop_C8_3f4741bac138,C8,13,6,1.0,1.0
1,crop_C8_4f61d31afbef,C8,11,2,1.0,1.0
2,crop_C8_f4106c2060df,C8,4,1,1.0,1.0
3,crop_D8_621c9e41edec,D8,17,10,1.0,1.0
4,crop_D8_c59003e258cd,D8,21,12,1.0,1.0


## 5. Per-well overlay visualizations (optional)

In [16]:
# ── PER-WELL OVERLAYS ─────────────────────────────────────────────────────────
if not CFG['write_overlays']:
    log('Overlays disabled.')
else:
    log('=' * 70)
    log('STEP 5: Generating per-well overlays ...')

    data_root = None
    _cands = []
    if CFG.get('data_root_override'):
        _cands.append(Path(CFG['data_root_override']))
    if nb03_manifest:
        _cands.append(Path(nb03_manifest.get('data_root', '')))
    _cands += [REPO_ROOT / 'data',
               Path.home() / 'Desktop' / 'spot_detection',
               Path.home() / 'Desktop']
    for cand in _cands:
        if cand.exists() and any(True for _ in cand.rglob('*.tiff')):
            data_root = cand; break

    if data_root is None:
        log('[warn] No TIFF root found — overlays skipped. Set data_root_override in CFG.')
    else:
        log(f'  data_root: {data_root}')

        _WL_RE   = re.compile(r'(\d{3})_nm', re.IGNORECASE)
        _WELL_RE = re.compile(r'(?<![A-Z0-9])([A-H](?:[1-9]|1[0-2]))(?![A-Z0-9])', re.IGNORECASE)

        img_rows = []
        for ext in ['*.tiff', '*.tif']:
            for f in data_root.rglob(ext):
                if f.name.startswith('._'): continue
                wm  = _WELL_RE.search(f.name)
                wlm = _WL_RE.search(f.name)
                if wm and wlm and 'BF' not in f.name.upper():
                    img_rows.append({'well_id': wm.group(1).upper(), 'wl': wlm.group(1), 'path': f})
        img_index = pd.DataFrame(img_rows).drop_duplicates(subset=['well_id', 'wl'])

        def _load_chan(well_id):
            for wl in ['638', '561']:
                row = img_index[(img_index['well_id'] == well_id) & (img_index['wl'] == wl)]
                if len(row) == 0: continue
                arr = np.squeeze(tifffile.imread(str(row.iloc[0]['path'])))
                if arr.ndim == 3: arr = arr.max(axis=0)
                return arr.astype(np.float32)
            return None

        def _pct(img, lo=1, hi=99):
            a, b = np.percentile(img, lo), np.percentile(img, hi)
            return np.clip((img - a) / max(b - a, 1e-8), 0, 1)

        wells = preds_df['well_id'].astype(str).unique().tolist()[:CFG['overlay_max_wells']]
        R = CFG['overlay_circle_radius_px']

        for well_id in wells:
            log(f'  Rendering {well_id} ...')
            img = _load_chan(well_id)
            if img is None:
                log(f'  [warn] No 638/561 TIFF for {well_id} — skip'); continue

            wdf  = preds_df[preds_df['well_id'].astype(str) == well_id]
            pos  = wdf[wdf['predicted_label'] == 1]

            # GT spots within this well (from QC eval)
            gt = pd.DataFrame()
            if len(qc_eval) > 0 and 'well_id' in qc_eval.columns:
                _gt = qc_eval[(qc_eval['well_id'].astype(str) == well_id) &
                              (qc_eval['target_binary'] == 1.0)]
                if len(_gt):
                    gt = _gt.merge(
                        preds_df[['union_id', 'union_medoid_well_y_px', 'union_medoid_well_x_px']],
                        on='union_id', how='left'
                    )

            fig, ax = plt.subplots(figsize=(9, 9))
            ax.imshow(_pct(img, CFG['overlay_lo_pct'], CFG['overlay_hi_pct']),
                      cmap='gray', origin='upper')

            for _, r in pos.iterrows():
                cy = r.get('union_medoid_well_y_px', float('nan'))
                cx = r.get('union_medoid_well_x_px', float('nan'))
                if np.isfinite(cy) and np.isfinite(cx):
                    ax.add_patch(plt.Circle((cx, cy), R,
                                 edgecolor='yellow', facecolor='none', lw=0.8, zorder=4))

            for _, r in gt.iterrows():
                cy = r.get('union_medoid_well_y_px', float('nan'))
                cx = r.get('union_medoid_well_x_px', float('nan'))
                if np.isfinite(cy) and np.isfinite(cx):
                    ax.add_patch(plt.Circle((cx, cy), R + 4,
                                 edgecolor='lime', facecolor='none', lw=1.5, zorder=5))

            ax.legend(handles=[
                mpatches.Patch(facecolor='none', edgecolor='yellow', lw=1,
                               label=f'Predicted pos ({len(pos):,})'),
                mpatches.Patch(facecolor='none', edgecolor='lime', lw=1.5,
                               label=f'GT annotation ({len(gt):,})'),
            ], loc='upper right', fontsize=8, framealpha=0.85)
            ax.set_title(
                f'Full-well inference | well={well_id} | model={PRIMARY_MODEL_ID}\n'
                f'threshold={DECISION_THRESHOLD:.4f} | '
                f'{len(pos):,}/{len(wdf):,} candidates positive',
                fontsize=9
            )
            ax.axis('off'); fig.tight_layout()
            op = REPORT_DIR / f'{RUN_ID}_overlay_{well_id}.png'
            fig.savefig(op, dpi=150, bbox_inches='tight')
            plt.show(); plt.close(fig)
            log(f'  Saved: {op.name}')

[2026-01-25 03:27:06 UTC] ======================================================================
[2026-01-25 03:27:06 UTC] STEP 5: Generating per-well overlays ...
[2026-01-25 03:27:06 UTC]   data_root: /Users/baghnatios/Desktop/spot_detection
[2026-01-25 03:27:06 UTC]   Rendering C8 ...
[2026-01-25 03:27:10 UTC]   Saved: nb07_20260125T032705Z_c3eff871_overlay_C8.png
[2026-01-25 03:27:10 UTC]   Rendering D8 ...


/var/folders/mf/blf3nbg52djf7wbt4080gd040000gn/T/ipykernel_10032/275977106.py:108: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close(fig)


[2026-01-25 03:27:15 UTC]   Saved: nb07_20260125T032705Z_c3eff871_overlay_D8.png


/var/folders/mf/blf3nbg52djf7wbt4080gd040000gn/T/ipykernel_10032/275977106.py:108: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close(fig)


## 6. Save outputs and manifest

In [17]:
# ── SAVE PREDICTIONS + QC TABLE ───────────────────────────────────────────────
log('=' * 70)
log('STEP 6: Saving outputs ...')

output_paths = {}

preds_path = safe_to_parquet(preds_df, TABLES_DIR / f'{RUN_ID}_fullwell_predictions.parquet')
output_paths['fullwell_predictions'] = str(preds_path)
log(f'  fullwell_predictions: {preds_path.name}  ({len(preds_df):,} rows)')

if len(qc_eval) > 0:
    qc_path = safe_to_parquet(qc_eval, REPORT_DIR / f'{RUN_ID}_qc_eval_table.parquet')
    output_paths['qc_eval_table'] = str(qc_path)
    log(f'  qc_eval_table: {qc_path.name}  ({len(qc_eval):,} rows)')

manifest = {
    'run_id':                    RUN_ID,
    'notebook_name':             '07_fullwell_inference.ipynb',
    'created_at_utc':            datetime.now(timezone.utc).isoformat(),
    'primary_model_id':          PRIMARY_MODEL_ID,
    'model_nb06_run_id':         NB06_RUN_ID,
    'nb03_run_id':               NB03_RUN_ID,
    'decision_threshold':        float(DECISION_THRESHOLD),
    'threshold_policy':          THRESHOLD_POLICY,
    'inference_registry_version': CFG['inference_registry_version'],
    'n_candidates_total':        int(len(preds_df)),
    'n_predicted_positive':      int(preds_df['predicted_label'].sum()),
    'wells':                     sorted(preds_df['well_id'].astype(str).unique().tolist()),
    'nb06_reported_test_metrics': NB06_REPORTED,
    'nb07_qc_metrics':           QC_RESULTS,
    'parity_tol':                1e-4,
    'inputs': {
        'candidate_union':    str(union_path),
        'candidate_features': str(features_path),
        'model':              str(cal_model_path),
        'feature_cols':       str(feature_cols_path),
        'clip_bounds':        str(clip_bounds_path),
        'train_medians':      str(train_medians_path),
    },
    'outputs': output_paths,
}
mp = write_json(manifest, MANIFEST_DIR / f'{RUN_ID}_fullwell_inference_manifest.json')
output_paths['manifest'] = str(mp)
log(f'  manifest: {mp.name}')

[2026-01-25 03:27:15 UTC] ======================================================================
[2026-01-25 03:27:15 UTC] STEP 6: Saving outputs ...
[2026-01-25 03:27:15 UTC]   fullwell_predictions: nb07_20260125T032705Z_c3eff871_fullwell_predictions.parquet  (5,955 rows)
[2026-01-25 03:27:15 UTC]   qc_eval_table: nb07_20260125T032705Z_c3eff871_qc_eval_table.parquet  (861 rows)
[2026-01-25 03:27:15 UTC]   manifest: nb07_20260125T032705Z_c3eff871_fullwell_inference_manifest.json


In [18]:
# ── EXIT CHECKS ───────────────────────────────────────────────────────────────
log('=' * 70)
log('EXIT CHECKS')

assert 'union_id'          in preds_df.columns, 'missing union_id'
assert 'prob_true_spot'    in preds_df.columns, 'missing prob_true_spot'
assert 'predicted_label'   in preds_df.columns, 'missing predicted_label'
assert 'decision_threshold' in preds_df.columns
assert preds_df['union_id'].is_unique, 'union_id not unique'
assert preds_df['prob_true_spot'].between(0, 1).all(), 'prob out of [0,1]'
assert set(preds_df['predicted_label'].unique()).issubset({0, 1}), 'bad predicted_label values'

log('All exit checks passed.')
log('')
log('=' * 70)
log('NOTEBOOK 07 COMPLETE')
log(f'  Model              : {PRIMARY_MODEL_ID} (NB06 run {NB06_RUN_ID})')
log(f'  Threshold          : {DECISION_THRESHOLD:.6f}  (policy={THRESHOLD_POLICY})')
log(f'  Candidates scored  : {len(preds_df):,}')
log(f'  Predicted positive : {int(preds_df["predicted_label"].sum()):,} '
    f'({100*preds_df["predicted_label"].mean():.2f}%)')
if QC_RESULTS:
    for sr, m in QC_RESULTS.items():
        log(f'  QC [{sr:12s}]  AP={m.get("avg_precision", float("nan")):.4f}  '
            f'F1={m.get("f1", float("nan")):.4f}  '
            f'AUC={m.get("roc_auc", float("nan")):.4f}')
    if 'test' in QC_RESULTS and NB06_REPORTED:
        ref = NB06_REPORTED
        log(f'  NB06 ref  [test ]  AP={ref.get("avg_precision", float("nan")):.4f}  '
            f'F1={ref.get("f1", float("nan")):.4f}  '
            f'AUC={ref.get("roc_auc", float("nan")):.4f}')
log(f'  Predictions saved  : {preds_path.name}')
log(f'  Manifest saved     : {mp.name}')
log('=' * 70)

[2026-01-25 03:27:15 UTC] ======================================================================
[2026-01-25 03:27:15 UTC] EXIT CHECKS
[2026-01-25 03:27:15 UTC] All exit checks passed.
[2026-01-25 03:27:15 UTC] 
[2026-01-25 03:27:15 UTC] ======================================================================
[2026-01-25 03:27:15 UTC] NOTEBOOK 07 COMPLETE
[2026-01-25 03:27:15 UTC]   Model              : lgbm_tuned (NB06 run nb06_20260124T172633Z_5cf12d26)
[2026-01-25 03:27:15 UTC]   Threshold          : 0.600000  (policy=max_f1)
[2026-01-25 03:27:15 UTC]   Candidates scored  : 5,955
[2026-01-25 03:27:15 UTC]   Predicted positive : 2,872 (48.23%)
[2026-01-25 03:27:15 UTC]   QC [test        ]  AP=1.0000  F1=1.0000  AUC=1.0000
[2026-01-25 03:27:15 UTC]   QC [calibration ]  AP=0.8233  F1=0.8485  AUC=0.9871
[2026-01-25 03:27:15 UTC]   QC [train       ]  AP=0.9669  F1=0.9624  AUC=0.9841
[2026-01-25 03:27:15 UTC]   NB06 ref  [test ]  AP=0.8142  F1=0.8315  AUC=0.8771
[2026-01-25 03:27:15 UTC]   

In [20]:
# ── RANDOM CROP VISUALIZATION — 3 crops per well, all 5 channels ─────────────
# Defines new crops on-the-fly (not from the crop registry).
# For each well: picks 3 random 256x256 windows that contain predicted spots,
# then shows all 5 barcode panels side-by-side with yellow circles on detections.

import numpy as np
import matplotlib.pyplot as plt
import tifffile
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
VIZ_CROP_SIZE_PX   = 256   # side length of each random crop window
VIZ_CROPS_PER_WELL = 3
VIZ_CIRCLE_RADIUS  = 8     # pixels
VIZ_LO_PCT, VIZ_HI_PCT = 1, 99
VIZ_SEED           = 42

# Panel definitions — matches NB03 CFG exactly
PANEL_TO_FOLDER_WL = {
    "C1_638": ("Cycle1_view", "638"),
    "C4_638": ("Cycle4_view", "638"),
    "C4_561": ("Cycle4_view", "561"),
    "C5_638": ("Cycle5_view", "638"),
    "C5_561": ("Cycle5_view", "561"),
}
PANEL_ORDER = ["C1_638", "C4_638", "C4_561", "C5_638", "C5_561"]

# ── Helpers ───────────────────────────────────────────────────────────────────
def _load_panel(data_root, well_id, panel_key):
    folder, wl = PANEL_TO_FOLDER_WL[panel_key]
    p = Path(data_root) / well_id / folder / f"{well_id}_common_Fluorescence_{wl}_nm_Ex.tiff"
    if not p.exists():
        return None
    arr = np.squeeze(tifffile.imread(str(p))).astype(np.float32)
    if arr.ndim == 3:
        arr = arr.max(axis=0)
    return arr

def _pct_clip(img, lo=1, hi=99):
    a, b = np.percentile(img, lo), np.percentile(img, hi)
    return np.clip((img - a) / max(b - a, 1e-8), 0, 1)

def _pick_crop_windows(pos_y, pos_x, H, W, crop_size, n_crops, rng):
    """
    Pick n_crops windows of size crop_size x crop_size that are:
      - fully within the image
      - contain at least one predicted positive spot
    Strategy: sample candidate top-left corners near predicted spots.
    """
    half = crop_size // 2
    # Candidate centres: jitter around each positive spot
    centres_y = np.clip(pos_y + rng.integers(-half, half, size=len(pos_y)), half, H - half - 1)
    centres_x = np.clip(pos_x + rng.integers(-half, half, size=len(pos_x)), half, W - half - 1)

    # Shuffle and deduplicate by grid cell (one crop per grid cell)
    idx = rng.permutation(len(centres_y))
    selected = []
    used_cells = set()
    for i in idx:
        cy, cx = int(centres_y[i]), int(centres_x[i])
        cell = (cy // crop_size, cx // crop_size)
        if cell in used_cells:
            continue
        used_cells.add(cell)
        y0, y1 = cy - half, cy + half
        x0, x1 = cx - half, cx + half
        # Clamp to image bounds
        y0, y1 = max(0, y0), min(H, y0 + crop_size)
        x0, x1 = max(0, x0), min(W, x0 + crop_size)
        if (y1 - y0) >= crop_size // 2 and (x1 - x0) >= crop_size // 2:
            selected.append((y0, y1, x0, x1))
        if len(selected) >= n_crops:
            break

    # Fallback: random windows anywhere in image if not enough spot-centered ones
    attempts = 0
    while len(selected) < n_crops and attempts < 200:
        attempts += 1
        cy = rng.integers(half, H - half)
        cx = rng.integers(half, W - half)
        y0, y1 = cy - half, cy + crop_size - half
        x0, x1 = cx - half, cx + crop_size - half
        y0, y1 = max(0, y0), min(H, y1)
        x0, x1 = max(0, x0), min(W, x1)
        selected.append((y0, y1, x0, x1))

    return selected[:n_crops]

# ── Main ──────────────────────────────────────────────────────────────────────
rng = np.random.default_rng(VIZ_SEED)

# data_root: reuse what was found in Step 5, or re-discover
try:
    _dr = data_root   # set by Step 5
except NameError:
    _dr = None
    for _cand in [Path.home() / "Desktop" / "spot_detection", Path.home() / "Desktop"]:
        if _cand.exists() and any(True for _ in _cand.rglob("*.tiff")):
            _dr = _cand; break

if _dr is None:
    print("[skip] data_root not found. Set data_root_override in CFG and re-run Step 5 first.")
else:
    wells = sorted(preds_df["well_id"].astype(str).unique().tolist())

    for well_id in wells:
        print(f"\n{'='*70}")
        print(f"Well {well_id} — loading all 5 panels ...")

        # Load all 5 panels for this well
        panels = {}
        for pk in PANEL_ORDER:
            img = _load_panel(_dr, well_id, pk)
            if img is not None:
                panels[pk] = img
                print(f"  {pk}: {img.shape}")
            else:
                print(f"  {pk}: NOT FOUND")

        if not panels:
            print(f"  [skip] No panels found for {well_id}")
            continue

        # Reference image shape (use first available panel)
        ref = next(iter(panels.values()))
        H, W = ref.shape

        # Get predicted positive spots for this well
        wdf = preds_df[preds_df["well_id"].astype(str) == well_id]
        pos = wdf[wdf["predicted_label"] == 1].copy()
        pos_y = pos["union_medoid_well_y_px"].dropna().values.astype(int)
        pos_x = pos["union_medoid_well_x_px"].dropna().values.astype(int)
        pos_y = np.clip(pos_y, 0, H - 1)
        pos_x = np.clip(pos_x, 0, W - 1)

        print(f"  {len(pos):,} predicted positives in well {well_id}")

        # Pick crop windows
        windows = _pick_crop_windows(pos_y, pos_x, H, W, VIZ_CROP_SIZE_PX,
                                     VIZ_CROPS_PER_WELL, rng)
        print(f"  {len(windows)} crop windows selected")

        for crop_idx, (y0, y1, x0, x1) in enumerate(windows):
            # Which spots fall inside this crop?
            in_crop = (pos_y >= y0) & (pos_y < y1) & (pos_x >= x0) & (pos_x < x1)
            cy_local = pos_y[in_crop] - y0
            cx_local = pos_x[in_crop] - x0
            n_spots  = int(in_crop.sum())

            n_panels = len(PANEL_ORDER)
            fig, axes = plt.subplots(1, n_panels, figsize=(4 * n_panels, 4.2),
                                     gridspec_kw={"wspace": 0.03})
            if n_panels == 1:
                axes = [axes]

            for ax, pk in zip(axes, PANEL_ORDER):
                if pk in panels:
                    crop_img = panels[pk][y0:y1, x0:x1]
                    ax.imshow(_pct_clip(crop_img, VIZ_LO_PCT, VIZ_HI_PCT),
                              cmap="gray", origin="upper")
                else:
                    ax.set_facecolor("black")
                    ax.text(0.5, 0.5, "missing", color="white",
                            ha="center", va="center", transform=ax.transAxes, fontsize=8)

                # Draw detection circles
                for cy, cx in zip(cy_local, cx_local):
                    ax.add_patch(plt.Circle((cx, cy), VIZ_CIRCLE_RADIUS,
                                 edgecolor="yellow", facecolor="none",
                                 lw=1.2, zorder=5))

                ax.set_title(pk, fontsize=9, pad=3)
                ax.set_xticks([]); ax.set_yticks([])

            fig.suptitle(
                f"Well {well_id} | crop {crop_idx + 1}/{len(windows)}  "
                f"[y={y0}:{y1}, x={x0}:{x1}]  |  "
                f"{n_spots} detected spot(s) in window  |  "
                f"model={PRIMARY_MODEL_ID}  thr={DECISION_THRESHOLD:.3f}",
                fontsize=9, y=1.01
            )
            fig.patch.set_facecolor("black")

            save_path = REPORT_DIR / f"{RUN_ID}_crop_viz_{well_id}_{crop_idx+1}.png"
            fig.savefig(save_path, dpi=160, bbox_inches="tight", facecolor="black")
            display(fig) 
            plt.close(fig)
            print(f"  Saved: {save_path.name}  ({n_spots} spots in window)")



Well C8 — loading all 5 panels ...
  C1_638: (7946, 7869)
  C4_638: (7946, 7869)
  C4_561: (7946, 7869)
  C5_638: (7946, 7869)
  C5_561: (7946, 7869)
  710 predicted positives in well C8
  3 crop windows selected


<Figure size 2000x420 with 5 Axes>

  Saved: nb07_20260125T032705Z_c3eff871_crop_viz_C8_1.png  (4 spots in window)


<Figure size 2000x420 with 5 Axes>

  Saved: nb07_20260125T032705Z_c3eff871_crop_viz_C8_2.png  (2 spots in window)


...

  Saved: nb07_20260125T032705Z_c3eff871_crop_viz_D8_1.png  (4 spots in window)


<Figure size 2000x420 with 5 Axes>

  Saved: nb07_20260125T032705Z_c3eff871_crop_viz_D8_2.png  (5 spots in window)


<Figure size 2000x420 with 5 Axes>

  Saved: nb07_20260125T032705Z_c3eff871_crop_viz_D8_3.png  (4 spots in window)


In [21]:
# ── DIAGNOSTIC: ON vs OFF channel ADU under predicted positive spots ──────────
import numpy as np
import pandas as pd
import tifffile
from pathlib import Path

PATCH_R = 4  # radius in pixels for mean ADU measurement

results = []

for well_id in sorted(preds_df["well_id"].astype(str).unique()):
    pos = preds_df[(preds_df["well_id"].astype(str) == well_id) &
                   (preds_df["predicted_label"] == 1)].copy()
    if len(pos) == 0:
        continue

    ys = pos["union_medoid_well_y_px"].values.astype(int)
    xs = pos["union_medoid_well_x_px"].values.astype(int)

    on_panels  = {"C8": ["C1_638","C4_638","C4_561"],
                  "D8": ["C5_638","C5_561","C1_638"]}
    off_panels = {"C8": ["C5_638","C5_561"],
                  "D8": ["C4_638","C4_561"]}

    for pk in PANEL_ORDER:
        img = _load_panel(_dr, well_id, pk)
        if img is None:
            continue
        H, W = img.shape
        patch_means = []
        for y, x in zip(ys, xs):
            y0, y1 = max(0, y-PATCH_R), min(H, y+PATCH_R+1)
            x0, x1 = max(0, x-PATCH_R), min(W, x+PATCH_R+1)
            patch_means.append(float(img[y0:y1, x0:x1].mean()))

        is_on = pk in on_panels.get(well_id, [])
        results.append({
            "well_id":    well_id,
            "panel":      pk,
            "on_off":     "ON" if is_on else "OFF",
            "mean_adu":   float(np.mean(patch_means)),
            "median_adu": float(np.median(patch_means)),
            "n_spots":    len(patch_means),
        })

diag_df = pd.DataFrame(results)
display(diag_df.sort_values(["well_id","on_off","panel"]))

print("\nSummary — mean ADU under predicted positives:")
display(diag_df.groupby(["well_id","on_off"])[["mean_adu","median_adu"]].mean().round(1))

print("\nON/OFF ratio per well (should be >> 1 if barcoding is working):")
for wid, g in diag_df.groupby("well_id"):
    on_mean  = g[g["on_off"]=="ON"]["mean_adu"].mean()
    off_mean = g[g["on_off"]=="OFF"]["mean_adu"].mean()
    ratio = on_mean / max(off_mean, 1e-6)
    status = "✅" if ratio > 1.5 else "⚠️"
    print(f"  {wid}: ON={on_mean:.1f}  OFF={off_mean:.1f}  ratio={ratio:.2f}  {status}")

,well_id,panel,on_off,mean_adu,median_adu,n_spots
4,C8,C5_561,OFF,1372.649772,1340.800110,710
3,C8,C5_638,OFF,1927.389051,1763.268311,710
0,C8,C1_638,ON,9945.859553,9200.197266,710
2,C8,C4_561,ON,4280.570115,3926.989014,710
1,C8,C4_638,ON,11281.415379,10426.646973,710
7,D8,C4_561,OFF,1227.314903,1208.027893,2162
6,D8,C4_638,OFF,2068.957413,1974.245483,2162
5,D8,C1_638,ON,8225.323094,8015.407227,2162
9,D8,C5_561,ON,5799.052780,5635.251465,2162
8,D8,C5_638,ON,12007.672792,12259.136230,2162



Summary — mean ADU under predicted positives:


mean_adu  median_adu
well_id on_off                      
C8      OFF       1650.0      1552.0
        ON        8502.6      7851.3
D8      OFF       1648.1      1591.1
        ON        8677.3      8636.6


ON/OFF ratio per well (should be >> 1 if barcoding is working):
  C8: ON=8502.6  OFF=1650.0  ratio=5.15  ✅
  D8: ON=8677.3  OFF=1648.1  ratio=5.26  ✅
